# Custom LLM Training Notebook (Google Colab Free Tier)

This notebook allows you to train your custom Decoder-only Transformer model from scratch using Colab's free GPU (typically a Tesla T4). 
It mounts your Google Drive so that training checkpoints are saved persistently. You can stop training at any time, and resume later from the exact same checkpoint.

### Step 1: Mount Google Drive
This ensures that checkpoints are saved directly to your Google Drive and won't be deleted when your Colab session expires.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Create Project Directories in Drive
We will set up a directory structure in Google Drive to hold our training files, datasets, and checkpoints.

In [ ]:
import os
project_dir = "/content/drive/MyDrive/CustomLLM"
os.makedirs(project_dir, exist_ok=True)
os.makedirs(os.path.join(project_dir, "custom_llm"), exist_ok=True)
print(f"Project workspace initialized in Drive: {project_dir}")

### Step 3: Install Required Libraries
We need PyTorch, Hugging Face `datasets` for streaming data, and `transformers` for the BPE tokenizer.

In [ ]:
!pip install torch datasets transformers

### Step 4: Write Custom LLM Source Codes
Let's write the code directly to Google Drive so that it matches our local setup. We will write `model.py`, `tokenizer.py`, `dataset.py`, and `train.py`.

In [ ]:
# Write model.py
model_code = """import math
import torch
import torch.nn as nn
from torch.nn import functional as F

class TransformerConfig:
    def __init__(
        self,
        vocab_size=50257,
        block_size=256,
        n_layer=6,
        n_head=6,
        n_embd=384,
        dropout=0.1,
        bias=True
    ):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        self.bias = bias

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.register_buffer(\"bias\", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if hasattr(F, 'scaled_dot_product_attention'):
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
            
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def get_num_params(self):
        n_params = sum(p.numel() for p in self.parameters())
        n_params -= self.transformer.wte.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
"""
with open(os.path.join(project_dir, "custom_llm", "model.py"), "w") as f:
    f.write(model_code)

print("model.py written to Google Drive.")

In [ ]:
# Write tokenizer.py (configured for Google Colab folders)
tokenizer_code = """import os
import json

class RobustTokenizer:
    def __init__(self, save_dir=\"/content/drive/MyDrive/CustomLLM/custom_llm/tokenizer_cache\"):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.tokenizer_type = \"char\"
        self.vocab = {}
        self.inverse_vocab = {}
        self.encoder = None
        try:
            from transformers import GPT2TokenizerFast
            if os.path.exists(os.path.join(save_dir, \"vocab.json\")):
                self.encoder = GPT2TokenizerFast.from_pretrained(save_dir)
                self.tokenizer_type = \"bpe\"
            else:
                self.encoder = GPT2TokenizerFast.from_pretrained(\"gpt2\")
                self.encoder.save_pretrained(save_dir)
                self.tokenizer_type = \"bpe\"
        except Exception as e:
            self.load_or_build_char_tokenizer()

    def load_or_build_char_tokenizer(self):
        vocab_path = os.path.join(self.save_dir, \"char_vocab.json\")
        if os.path.exists(vocab_path):
            with open(vocab_path, \"r\", encoding=\"utf-8\") as f:
                self.vocab = json.load(f)
            self.inverse_vocab = {int(v): k for k, v in self.vocab.items()}
        else:
            chars = (\" \\n\\r\\t!\\\"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\\\\\\"\\]^_`abcdefghijklmnopqrstuvwxyz{|}~\"
                     \"०१२३४५६७८९अआइईउऊऋएऐओऔकखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहािीुूृेैोौ्ँंः\"
                     \"౦౧౨౩౪౫౬౭౮౯అఆఇఈఉఉఋఎఏఐఒఓఔకఖగఘఙచఛజఝఞటఠడఢణతథదధనపఫబభమయరలవశషసహానీుూృెేైొోౌ్\")
            unique_chars = sorted(list(set(chars)))
            self.vocab = {ch: i for i, ch in enumerate(unique_chars)}
            self.vocab[\"<|endoftext|>\"] = len(self.vocab)
            self.inverse_vocab = {i: ch for ch, i in self.vocab.items()}
            with open(vocab_path, \"w\", encoding=\"utf-8\") as f:
                json.dump(self.vocab, f, ensure_ascii=False, indent=2)

    @property
    def vocab_size(self):
        return self.encoder.vocab_size if self.tokenizer_type == \"bpe\" else len(self.vocab)

    def encode(self, text):
        return self.encoder.encode(text) if self.tokenizer_type == \"bpe\" else [self.vocab.get(ch, self.vocab.get(\" \", 0)) for ch in text]

    def decode(self, ids):
        return self.encoder.decode(ids) if self.tokenizer_type == \"bpe\" else \"\".join([self.inverse_vocab.get(idx, \"\") for idx in ids])
"""
with open(os.path.join(project_dir, "custom_llm", "tokenizer.py"), "w") as f:
    f.write(tokenizer_code)

print("tokenizer.py written to Google Drive.")

In [ ]:
# Write dataset.py
dataset_code = """import os
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, tokenizer, block_size=256, split=\"train\", cache_dir=\"/content/drive/MyDrive/CustomLLM/custom_llm/dataset_cache\"):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        text_data = \"\"
        local_txt_path = os.path.join(cache_dir, f\"{split}_data.txt\")
        if os.path.exists(local_txt_path):
            with open(local_txt_path, \"r\", encoding=\"utf-8\") as f: text_data = f.read()
        else:
            try:
                from datasets import load_dataset
                dataset = load_dataset(\"wikitext\", \"wikitext-2-raw-v1\", split=split)
                text_data = \"\\n\".join(dataset[\"text\"])
                with open(local_txt_path, \"w\", encoding=\"utf-8\") as f: f.write(text_data)
            except Exception as e:
                text_data = self.generate_fallback_data(split)
                with open(local_txt_path, \"w\", encoding=\"utf-8\") as f: f.write(text_data)
        self.tokens = torch.tensor(self.tokenizer.encode(text_data), dtype=torch.long)

    def generate_fallback_data(self, split):
        stories = [\"Once upon a time there was a small bird that loved to fly.\",
                   \"Artificial intelligence is training models to think and predict next tokens.\",
                   \"Deep learning uses PyTorch networks to compute backpropagation and optimize weights.\"]
        repeated = [s for _ in range(500) for s in stories]
        return \"\\n\".join(repeated)

    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):
        chunk = self.tokens[idx : idx + self.block_size + 1]
        return chunk[:-1], chunk[1:]
"""
with open(os.path.join(project_dir, "custom_llm", "dataset.py"), "w") as f:
    f.write(dataset_code)

print("dataset.py written to Google Drive.")

In [ ]:
# Write train.py
train_code = """import os
import sys
import time
import json
import torch
from torch.utils.data import DataLoader

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from model import GPT, TransformerConfig
from tokenizer import RobustTokenizer
from dataset import TextDataset

def save_checkpoint(model, optimizer, epoch, step, loss, path):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'step': step,
        'loss': loss,
        'config': model.config
    }, path)
    print(f\"\\n[CHECKPOINT] Saved checkpoint to {path} at Epoch {epoch}, Step {step}, Loss {loss:.4f}\")

def load_checkpoint(path, device):
    return torch.load(path, map_location=device) if os.path.exists(path) else None

def update_json_log(log_path, epoch, step, loss, speed):
    history = []
    if os.path.exists(log_path):
        try:
            with open(log_path, \"r\") as f: history = json.load(f)
        except: pass
    history.append({\"epoch\": epoch, \"step\": step, \"loss\": float(loss), \"speed\": float(speed), \"timestamp\": time.time()})
    with open(log_path, \"w\") as f: json.dump(history, f, indent=2)

def train(epochs=10, batch_size=32, block_size=128, lr=6e-4, checkpoint_path=\"/content/drive/MyDrive/CustomLLM/checkpoint.pt\", log_path=\"/content/drive/MyDrive/CustomLLM/train_log.json\"):
    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"
    print(f\"Training on {device}\")
    tokenizer = RobustTokenizer()
    dataset = TextDataset(tokenizer, block_size=block_size, split=\"train\")
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    checkpoint = load_checkpoint(checkpoint_path, device)
    if checkpoint:
        model = GPT(checkpoint['config'])
        model.load_state_dict(checkpoint['model_state_dict'])
        start_epoch, start_step, last_loss = checkpoint['epoch'], checkpoint['step'], checkpoint['loss']
        print(f\"Resumed from Epoch {start_epoch}, Step {start_step} (Loss {last_loss:.4f})\")
    else:
        model = GPT(TransformerConfig(vocab_size=tokenizer.vocab_size, block_size=block_size, n_layer=4, n_head=4, n_embd=256))
        start_epoch, start_step, last_loss = 0, 0, 0.0
        
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    if checkpoint: optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler = torch.cuda.amp.GradScaler(enabled=(device == \"cuda\"))
    
    total_steps = start_step
    model.train()
    try:
        for epoch in range(start_epoch, epochs):
            print(f\"\\n--- Epoch {epoch+1}/{epochs} ---\")
            for step, (x, y) in enumerate(dataloader):
                t0 = time.time()
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(device == \"cuda\")):
                    logits, loss = model(x, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                dt = time.time() - t0
                tok_s = (batch_size * block_size) / dt
                total_steps += 1
                print(f\"Step {total_steps} | Loss: {loss.item():.4f} | Speed: {tok_s:.0f} tok/s\", end=\"\\r\")
                update_json_log(log_path, epoch, total_steps, loss.item(), tok_s)
                if total_steps % 100 == 0: save_checkpoint(model, optimizer, epoch, total_steps, loss.item(), checkpoint_path)
    except KeyboardInterrupt:
        save_checkpoint(model, optimizer, epoch, total_steps, loss.item(), checkpoint_path)
        print(\"Paused.\")
    save_checkpoint(model, optimizer, epochs, total_steps, loss.item(), checkpoint_path)
if __name__ == '__main__':
    train()
"""
with open(os.path.join(project_dir, "custom_llm", "train.py"), "w") as f:
    f.write(train_code)

print("train.py written to Google Drive.")

### Step 5: Start or Resume Training
We will run `train.py` from here. If you stop the execution (by clicking the Stop button in Colab), the script will catch the interrupt, write a checkpoint, and exit gracefully. Running this block again will automatically pick up where it left off!

In [ ]:
%cd /content/drive/MyDrive/CustomLLM/custom_llm
!python train.py --checkpoint_path "/content/drive/MyDrive/CustomLLM/checkpoint.pt" --log_path "/content/drive/MyDrive/CustomLLM/train_log.json" --epochs 20 --batch_size 32